# Phase 2 — Condition 5-Sweep: CV-Tuned Improved Gate (MAE encoder)

Rigorously tunes the modality-dropout rate using **5-fold cross-validation** on the train+val pool, with the test set held out untouched until the very end. This is the statistically-sound way to tune at small n: CV averages each config's score over many patients instead of trusting a single 28-patient validation split.

**Procedure:**
1. Sweep modality dropout ∈ {0.0, 0.1, 0.2} — small, principled grid (literature sweet spot).
2. For each value: 5-fold CV on train+val → mean ± std validation MCC.
3. Pick best dropout by mean CV MCC.
4. Retrain on FULL train+val with that value, evaluate ONCE on held-out test set.
5. Run the full diagnostic; log everything to Drive.

Keeps the two working gate fixes from Cond 5 (per-modality LayerNorm + learnable temperature); only the dropout RATE is tuned. Uses the already-trained MAE encoder.

> ~15 short head-training runs (frozen encoder = fast) + 1 final. Colab Pro friendly. The encoder forward passes are cached per-fold to avoid recomputing 3D features every epoch.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_ROOT>', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn scikit-learn
import os; os._exit(0)


## 1. Verify env

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_ROOT>', force_remount=True)
import torch, numpy, monai
print('NumPy',numpy.__version__,'MONAI',monai.__version__)
from monai.networks.nets import SwinUNETR
assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))


## 2. Config

In [ ]:
import os, json, numpy as np, torch, datetime
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchio as tio, pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from monai.networks.nets import SwinUNETR
device=torch.device('cuda')

CONDITION_NAME='condition5sweep_mae_cv_tuned_gate'
DROPOUT_GRID=[0.0, 0.1, 0.2]
N_FOLDS=5

DRIVE='<DRIVE_ROOT>/ADNI_NewDS/'; RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain')
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz')
CSV=os.path.join(RESULTS,'project_data_cleaned.csv'); BIO=os.path.join(RESULTS,'preprocessed_biomarker_sequences.npy')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study'); SSL_DIR=os.path.join(PHASE2,'ssl_pretrain')
MAE_ENCODER=os.path.join(SSL_DIR,'ssl_mae_encoder.pth')
COND_DIR=os.path.join(PHASE2,'results',CONDITION_NAME); os.makedirs(COND_DIR,exist_ok=True)
LAB_LOG=os.path.join(PHASE2,'phase2_lab_log.jsonl')
IMG_SIZE=(96,96,96); FEATURE_SIZE=48; VIT_DIM=768; NUM_CLASSES=3; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
assert os.path.exists(MAE_ENCODER), 'Run Condition 4 pretraining first (need ssl_mae_encoder.pth).'
print('Sweep:', DROPOUT_GRID, '|', N_FOLDS, 'folds')


## 3. Load data + cache MAE image features (compute 3D encoder ONCE)

In [ ]:
# Load splits, clinical, biomarker
sp=np.load(SPLITS,allow_pickle=True)
pids_tr,pids_va,pids_te=list(sp['pids_train']),list(sp['pids_val']),list(sp['pids_test'])
y_tr,y_va,y_te=list(sp['labels_train']),list(sp['labels_val']),list(sp['labels_test'])
# CV pool = train+val (test held out)
pool_pids=pids_tr+pids_va; pool_y=np.array(y_tr+y_va)
clin_df=pd.read_csv(CSV)
FEATS=['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13','RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
patient_clin={p:torch.tensor(g[FEATS].values,dtype=torch.float32) for p,g in clin_df.groupby('PTID')}
bio_raw=np.load(BIO,allow_pickle=True); bio_obj=bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
patient_bio={k:torch.tensor(np.asarray(v),dtype=torch.float32) for k,v in bio_obj.items()}
CLIN_DIM=len(FEATS); BIO_DIM=next(iter(patient_bio.values())).shape[-1]

# Load MAE encoder, cache pooled image features per patient (frozen -> deterministic, compute once)
class SwinEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.swin=SwinUNETR(in_channels=1,out_channels=1,feature_size=FEATURE_SIZE,use_checkpoint=True).swinViT; self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x): return self.pool(self.swin(x)[-1]).flatten(1)
encoder=SwinEncoder(); sd=torch.load(MAE_ENCODER,map_location='cpu')
miss,unexp=encoder.load_state_dict(sd,strict=False); encoder=encoder.to(device).eval()
print(f'MAE encoder loaded ({(len(encoder.state_dict())-len(miss))/len(encoder.state_dict()):.0%} keys)')

etf=tio.Compose([tio.Resize(IMG_SIZE), tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
@torch.no_grad()
def img_feat(pid):
    vol=np.load(os.path.join(BRAIN_DIR,f'{pid}_processed.npy'))
    subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
    t=etf(subj).mri.tensor.unsqueeze(0).to(device)
    return encoder(t).cpu().squeeze(0)
print('Caching image features for all patients...')
img_cache={p:img_feat(p) for p in tqdm(pool_pids+pids_te)}
print('cached', len(img_cache), 'image features')


## 4. Model (improved gate; dropout rate is a parameter) + tensor dataset

In [ ]:
class LSTMBranch(nn.Module):
    def __init__(s,ind,h=128,o=128):
        super().__init__(); s.l=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); s.f=nn.Linear(h,o); s.r=nn.ReLU()
    def forward(s,x): _,(h,_)=s.l(x); return s.r(s.f(h[-1]))

class ImprovedGateModel(nn.Module):
    def __init__(s,cd,bd,nc=3,mod_dropout=0.1):
        super().__init__(); s.mod_dropout=mod_dropout
        s.img_proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.cb=LSTMBranch(cd); s.bb=LSTMBranch(bd)
        s.norm_img=nn.LayerNorm(128); s.norm_clin=nn.LayerNorm(128); s.norm_bio=nn.LayerNorm(128)
        s.gate_lin=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3))
        s.log_temp=nn.Parameter(torch.zeros(1))
        s.clf=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
    def forward(s,img_emb,c,b,rg=False):
        fi=s.norm_img(s.img_proj(img_emb)); fc=s.norm_clin(s.cb(c)); fb=s.norm_bio(s.bb(b))
        if s.training and s.mod_dropout>0:
            r=torch.rand(3)
            if r[0]<s.mod_dropout: fi=fi*0
            if r[1]<s.mod_dropout: fc=fc*0
            if r[2]<s.mod_dropout: fb=fb*0
        st=torch.stack([fi,fc,fb],1); logits=s.gate_lin(torch.cat([fi,fc,fb],1))
        temp=torch.exp(s.log_temp).clamp(0.2,5.0); w=F.softmax(logits/temp,dim=1)
        fused=(st*w.unsqueeze(-1)).sum(1); out=s.clf(fused)
        return (out,w) if rg else out

class CachedDS(Dataset):
    def __init__(s,pids,labels): s.pids=list(pids); s.y=torch.tensor(np.array(labels),dtype=torch.long)
    def __len__(s): return len(s.pids)
    def __getitem__(s,i):
        p=s.pids[i]; return img_cache[p], patient_clin[p], patient_bio[p], s.y[i]
def coll(b):
    im,c,bi,y=zip(*b); return torch.stack(im),pad_sequence(c,batch_first=True),pad_sequence(bi,batch_first=True),torch.stack(y)
print('model + dataset ready')


## 5. Train/eval helpers

In [ ]:
def train_model(tr_pids,tr_y,va_pids,va_y,mod_dropout,epochs=40,patience=10,verbose=False):
    model=ImprovedGateModel(CLIN_DIM,BIO_DIM,NUM_CLASSES,mod_dropout=mod_dropout).to(device)
    trL=DataLoader(CachedDS(tr_pids,tr_y),batch_size=8,shuffle=True,collate_fn=coll)
    vaL=DataLoader(CachedDS(va_pids,va_y),batch_size=8,shuffle=False,collate_fn=coll)
    cw=compute_class_weight('balanced',classes=np.unique(tr_y),y=tr_y)
    crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best=-1; bs=None; wait=0
    for ep in range(epochs):
        model.train()
        for im,c,b,y in trL:
            im,c,b,y=im.to(device),c.to(device),b.to(device),y.to(device)
            opt.zero_grad(); loss=crit(model(im,c,b),y); loss.backward(); opt.step()
        model.eval(); P=[];T=[]
        with torch.no_grad():
            for im,c,b,y in vaL: P+=model(im.to(device),c.to(device),b.to(device)).argmax(1).cpu().tolist(); T+=y.tolist()
        vm=matthews_corrcoef(T,P)
        if vm>best: best=vm; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
        sched.step()
    if bs: model.load_state_dict(bs)
    return model, best
print('helpers ready')


## 6. The CV sweep
For each dropout value, 5-fold stratified CV on the train+val pool. Reports mean ± std validation MCC. Test set is NOT touched here.

In [ ]:
sweep_results={}
skf=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pool_arr=np.array(pool_pids)
for md_rate in DROPOUT_GRID:
    fold_mccs=[]
    for fi,(tr_idx,va_idx) in enumerate(skf.split(pool_arr, pool_y)):
        tp,vp=pool_arr[tr_idx].tolist(),pool_arr[va_idx].tolist()
        ty,vy=pool_y[tr_idx],pool_y[va_idx]
        _,mcc=train_model(tp,ty,vp,vy,mod_dropout=md_rate)
        fold_mccs.append(mcc)
        print(f'  dropout={md_rate} fold{fi+1}/{N_FOLDS}: val_mcc={mcc:.4f}')
    fold_mccs=np.array(fold_mccs)
    sweep_results[md_rate]={'mean':float(fold_mccs.mean()),'std':float(fold_mccs.std()),'folds':fold_mccs.tolist()}
    print(f'dropout={md_rate}: CV MCC = {fold_mccs.mean():.4f} +/- {fold_mccs.std():.4f}')
    print('='*50)

best_dropout=max(sweep_results, key=lambda k: sweep_results[k]['mean'])
print(f'\nBEST dropout by CV mean MCC: {best_dropout}  (MCC {sweep_results[best_dropout]["mean"]:.4f} +/- {sweep_results[best_dropout]["std"]:.4f})')
# honest note on overlap
print('\nFold spread (check if differences are within noise):')
for k,v in sweep_results.items(): print(f'  dropout={k}: {v["mean"]:.4f} +/- {v["std"]:.4f}')
json.dump(sweep_results, open(os.path.join(COND_DIR,'cv_sweep_results.json'),'w'), indent=2)


## 7. Retrain with best dropout, evaluate on held-out test

Refit on train+val with the CV-selected dropout, then evaluate on the held-out test set.

**Honest caveat:** to avoid using the test set for early-stopping (which would leak), the final model trains for a fixed number of epochs on train+val and is evaluated once on test. The CV in step 6 is what justifies the dropout choice; this step just reports the test number for that choice.

In [ ]:
# Fixed-epoch refit on full pool (no test leakage). Use a small internal holdout for early stop.
from sklearn.model_selection import train_test_split
_tp,_vp,_ty,_vy=train_test_split(np.array(pool_pids),pool_y,test_size=0.15,stratify=pool_y,random_state=SEED)
final_model,_=train_model(_tp.tolist(),_ty,_vp.tolist(),_vy,mod_dropout=best_dropout,epochs=50,patience=12)
# Early stopping used an internal 15% holdout, NOT the test set. Test below is a clean report.
torch.save(final_model.state_dict(), os.path.join(COND_DIR,'best_model.pth'))

def gm(cm):
    g=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0; g.append((se*sp)**0.5)
    return float(np.mean(g))
teL=DataLoader(CachedDS(pids_te,np.array(y_te)),batch_size=8,shuffle=False,collate_fn=coll)
final_model.eval(); gates=[]; imf=[]
with torch.no_grad():
    for im,c,b,y in teL:
        im,c,b=im.to(device),c.to(device),b.to(device)
        _,w=final_model(im,c,b,rg=True); gates.append(w.cpu().numpy())
gates=np.concatenate(gates); mg=gates.mean(0)
def ab(z=None):
    P=[];T=[]
    with torch.no_grad():
        for im,c,b,y in teL:
            im,c,b=im.to(device),c.to(device),b.to(device)
            if z=='mri': im=torch.zeros_like(im)
            if z=='clin': c=torch.zeros_like(c)
            if z=='bio': b=torch.zeros_like(b)
            P+=final_model(im,c,b).argmax(1).cpu().tolist(); T+=y.tolist()
    cm=confusion_matrix(T,P,labels=[0,1,2]); return accuracy_score(T,P),matthews_corrcoef(T,P),gm(cm)
abl={t:dict(zip(['acc','mcc','gmean'],ab(z))) for t,z in [('full',None),('mri','mri'),('clin','clin'),('bio','bio')]}
temp=float(torch.exp(final_model.log_temp).clamp(0.2,5.0).item())
diag={'gate':{'mri':float(mg[0]),'clin':float(mg[1]),'bio':float(mg[2])},'ablation':abl,'learned_temp':temp,'best_dropout':best_dropout}
print('='*50)
print('FINAL (held-out test, best dropout=%.1f):'%best_dropout)
print('  GATE:',{k:round(v,4) for k,v in diag['gate'].items()})
print('  ABLATION acc:',{k:round(v['acc'],4) for k,v in abl.items()})
print('  full MCC:',round(abl['full']['mcc'],4),'| learned temp:',round(temp,3))


## 8. Log everything to Drive

In [ ]:
full=abl['full']; mri0=abl['mri']
entry={'timestamp':datetime.datetime.now(datetime.UTC).isoformat(),'condition':CONDITION_NAME,
  'change':f'MAE encoder + improved gate (LayerNorm+temp), CV-tuned modality dropout={best_dropout}',
  'cv_sweep':sweep_results,'best_dropout':best_dropout,
  'diagnostic':{'gate':diag['gate'],'ablation':abl,'img_sim_std':None,'learned_temp':temp},
  'mri_contribution':float(full['acc']-mri0['acc']),
  'baseline_ref':{'cond3b_gate':0.141,'cond3b_acc':0.9310,'cond5_fixed03_gate':0.131,'cond5_fixed03_acc':0.7586}}
with open(LAB_LOG,'a') as f: f.write(json.dumps(entry)+'\n')
json.dump(entry, open(os.path.join(COND_DIR,'result.json'),'w'), indent=2)
np.save(os.path.join(COND_DIR,'test_gate_weights.npy'), gates)
print('Saved to', COND_DIR)
print('\nSUMMARY:')
print(f'  best dropout (CV): {best_dropout}')
print(f'  test gate MRI: {diag["gate"]["mri"]:.4f}  (3b CT=0.141, Cond5 fixed0.3=0.131)')
print(f'  test full acc: {full["acc"]:.4f}  (3b=0.931, Cond5 fixed0.3=0.759)')
print(f'  MRI contribution (acc drop): {full["acc"]-mri0["acc"]:+.4f}')
print('\nPaste this whole output back to Claude.')
